Imports

In [ ]:
# =========================
# REPRODUCIBILITY + IMPORTS
# =========================
from pathlib import Path
import os, random, json, csv, logging
import numpy as np
from typing import Dict, List
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import torchvision.transforms as T
from PIL import Image

from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix

# =========================
# LOGGING
# =========================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("training_multihead_stable.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# =========================
# CONSTANTS
# =========================
SEED = 42
CROP_MARGIN = 0.0

# Reduced augmentation to avoid unstable validation flipping
RANDOM_CROP_SCALE = (0.95, 1.0)
RANDOM_CROP_RATIO = (0.98, 1.02)
ROTATION_DEGREES = 2
TRANSLATE = (0.01, 0.01)
SCALE_RANGE = (0.99, 1.01)

GRAY_MEAN = (0.5,)
GRAY_STD = (0.25,)

GRADIENT_CLIP_MAX_NORM = 1.0
VALIDATION_RESIZE_FACTOR = 1.10

# =========================
# SEED
# =========================
def seed_everything(seed=42, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    logger.info(f"Set random seed to {seed}")

seed_everything(SEED)

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

# =========================
# CONFIG
# =========================
class Config:
    PROCESSED_ROOT = Path("../../data/processed/SHAMPOOBLADEINTRAY_COMPLETEV2/gray")
    INDEX_CSV = PROCESSED_ROOT / "index.csv"

    TRAIN_SPLIT = "train"
    TRAIN_LABELS_JSON = Path("../../data/labels/SHAMPOOBLADEINTRAY_COMPLETEV2/gray/train.json")
    VAL_LABELS_JSON = Path("../../data/labels/SHAMPOOBLADEINTRAY_COMPLETEV2/gray/val.json")

    MODEL_OUT_PATH = Path("../../models/classifier/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_multihead_stable/model.pt")
    CKPT_DIR = Path("../../models/classifier/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_multihead_stable/checkpoints")

    IMAGE_MODE = "gray"

    SPATIAL_CLASSES = ["isolated", "overlap"]
    THREAT_CLASSES = ["non_contraband", "contraband"]

    BATCH_SIZE = 8
    IMAGE_SIZE = 1024
    EPOCHS = 30

    LR_HEAD = 5e-4
    LR_FULL = 2e-5
    WEIGHT_DECAY = 1e-3
    UNFREEZE_EPOCH = 1

    SCHEDULER_PATIENCE = 4
    SCHEDULER_FACTOR = 0.7
    SCHEDULER_MIN_LR = 1e-7

    EARLY_STOPPING_PATIENCE = 8

    USE_AMP = True
    NUM_WORKERS = 2

    # Start fresh. Do not continue from unstable checkpoint.
    RESUME_TRAINING = False
    RESUME_CHECKPOINT_PATH = CKPT_DIR / f"{TRAIN_SPLIT}_best_checkpoint.pt"

    @classmethod
    def create_dirs(cls):
        cls.MODEL_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
        cls.CKPT_DIR.mkdir(parents=True, exist_ok=True)

config = Config()
config.create_dirs()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

# =========================
# LOAD INDEX CSV
# =========================
def read_index_csv(index_csv_path: Path) -> List[Dict]:
    rows = []
    with open(index_csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        rows.extend(reader)

    if not rows:
        raise ValueError(f"index.csv is empty: {index_csv_path}")

    logger.info(f"Read {len(rows)} rows from {index_csv_path}")
    return rows

index_rows = read_index_csv(config.INDEX_CSV)

# =========================
# LOAD LABEL JSON AS TWO HEADS
# =========================
def load_label_map(label_path: Path) -> Dict:
    data = json.load(open(label_path, "r"))

    out = {}
    bad = []

    for item in data:
        filepath = item.get("image", "").replace("\\", "/").strip()
        if not filepath:
            bad.append(("missing image", item))
            continue

        fname = Path(filepath).name

        overlap = int(item.get("overlap", 0))
        isolated = int(item.get("isolated", 0))
        contraband = int(item.get("contraband", 0))
        non_contraband = int(item.get("non_contraband", 0))

        if overlap == 1 and isolated == 0:
            spatial_label = 1
        elif overlap == 0 and isolated == 1:
            spatial_label = 0
        else:
            bad.append((filepath, f"bad spatial labels: overlap={overlap}, isolated={isolated}"))
            continue

        if contraband == 1 and non_contraband == 0:
            threat_label = 1
        elif contraband == 0 and non_contraband == 1:
            threat_label = 0
        else:
            bad.append((filepath, f"bad threat labels: contraband={contraband}, non_contraband={non_contraband}"))
            continue

        labels = {
            "spatial": spatial_label,
            "threat": threat_label,
        }

        out[filepath] = labels
        out[fname] = labels

    if bad:
        logger.warning(f"Bad label rows: {len(bad)}")
        for x in bad[:20]:
            logger.warning(x)
        raise ValueError(f"Found {len(bad)} invalid label rows")

    logger.info(f"Loaded {len(out)} label entries from {label_path}")
    return out

train_label_map = load_label_map(config.TRAIN_LABELS_JSON)
val_label_map = load_label_map(config.VAL_LABELS_JSON)

# =========================
# IMAGE HELPERS
# =========================
def crop_to_tray_interior(img: Image.Image) -> Image.Image:
    if CROP_MARGIN <= 0:
        return img

    w, h = img.size
    return img.crop((
        int(w * CROP_MARGIN),
        int(h * CROP_MARGIN),
        int(w * (1 - CROP_MARGIN)),
        int(h * (1 - CROP_MARGIN)),
    ))

class RandomBorderZero:
    def __init__(self, p=0.15, min_frac=0.02, max_frac=0.04):
        self.p = p
        self.min_frac = min_frac
        self.max_frac = max_frac

    def __call__(self, x):
        if torch.rand(1).item() > self.p:
            return x

        _, h, w = x.shape
        frac = float(torch.empty(1).uniform_(self.min_frac, self.max_frac))
        bx = int(w * frac)
        by = int(h * frac)

        x = x.clone()
        x[:, :by, :] = 0
        x[:, h - by:, :] = 0
        x[:, :, :bx] = 0
        x[:, :, w - bx:] = 0
        return x

# =========================
# DATASET
# =========================
class ProcessedSplitDataset(Dataset):
    def __init__(self, index_rows, processed_root, split, label_map, transform=None):
        self.processed_root = processed_root
        self.split = split.lower()
        self.transform = transform

        self.filepaths = []
        self.spatial_labels = []
        self.threat_labels = []

        missing = []

        for r in index_rows:
            fp = (r.get("filepath") or "").replace("\\", "/").strip()
            row_split = (r.get("split") or "").strip().lower()

            if row_split != self.split:
                continue

            fname = Path(fp).name
            label = label_map.get(fp, label_map.get(fname, None))

            if label is None:
                missing.append(fp)
                continue

            img_path = self.processed_root / fp
            if not img_path.exists():
                missing.append(str(img_path))
                continue

            self.filepaths.append(fp)
            self.spatial_labels.append(int(label["spatial"]))
            self.threat_labels.append(int(label["threat"]))

        if missing:
            raise KeyError(f"Missing labels/files: {len(missing)} examples. First few: {missing[:10]}")

        logger.info(f"Dataset {split}: {len(self.filepaths)} samples loaded")

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        rel_path = self.filepaths[idx]
        img_path = self.processed_root / rel_path

        img = Image.open(img_path).convert("L" if config.IMAGE_MODE == "gray" else "RGB")
        img = crop_to_tray_interior(img)

        if self.transform:
            img = self.transform(img)

        spatial_y = torch.tensor(self.spatial_labels[idx], dtype=torch.long)
        threat_y = torch.tensor(self.threat_labels[idx], dtype=torch.long)

        return img, spatial_y, threat_y

# =========================
# TRANSFORMS
# =========================
in_channels = 1 if config.IMAGE_MODE == "gray" else 3

train_transform = T.Compose([
    T.RandomResizedCrop(
        config.IMAGE_SIZE,
        scale=RANDOM_CROP_SCALE,
        ratio=RANDOM_CROP_RATIO,
    ),
    T.RandomAffine(
        degrees=ROTATION_DEGREES,
        translate=TRANSLATE,
        scale=SCALE_RANGE,
    ),
    T.ToTensor(),
    T.Normalize(GRAY_MEAN, GRAY_STD),
    RandomBorderZero(),
])

val_transform = T.Compose([
    T.Resize(int(config.IMAGE_SIZE * VALIDATION_RESIZE_FACTOR)),
    T.CenterCrop(config.IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(GRAY_MEAN, GRAY_STD),
])

# =========================
# CREATE DATASETS
# =========================
train_ds = ProcessedSplitDataset(
    index_rows=index_rows,
    processed_root=config.PROCESSED_ROOT,
    split=config.TRAIN_SPLIT,
    label_map=train_label_map,
    transform=train_transform,
)

val_ds = ProcessedSplitDataset(
    index_rows=index_rows,
    processed_root=config.PROCESSED_ROOT,
    split="val",
    label_map=val_label_map,
    transform=val_transform,
)

logger.info(f"Train spatial distribution: {Counter(train_ds.spatial_labels)}")
logger.info(f"Train threat distribution:  {Counter(train_ds.threat_labels)}")
logger.info(f"Val spatial distribution:   {Counter(val_ds.spatial_labels)}")
logger.info(f"Val threat distribution:    {Counter(val_ds.threat_labels)}")

# =========================
# CLASS WEIGHTS - SOFTENED
# =========================
def make_class_weights(labels, num_classes=2):
    counts = np.bincount(labels, minlength=num_classes)
    counts = np.maximum(counts, 1)

    weights = counts.sum() / (num_classes * counts)

    # soften imbalance strength
    weights = np.sqrt(weights)

    # prevent over-correction
    weights = np.clip(weights, 0.9, 1.2)

    return torch.tensor(weights, dtype=torch.float32).to(device)

spatial_weights = make_class_weights(train_ds.spatial_labels, 2)
threat_weights = make_class_weights(train_ds.threat_labels, 2)

logger.info(f"spatial_weights: {spatial_weights.tolist()}")
logger.info(f"threat_weights: {threat_weights.tolist()}")

criterion_spatial = nn.CrossEntropyLoss(weight=spatial_weights, label_smoothing=0.05)
criterion_threat = nn.CrossEntropyLoss(weight=threat_weights, label_smoothing=0.05)

# =========================
# LOADERS
# =========================
train_loader = DataLoader(
    train_ds,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)

x, spatial_y, threat_y = next(iter(train_loader))
logger.info(f"Batch image shape: {x.shape}")
logger.info(f"Batch spatial label shape: {spatial_y.shape}")
logger.info(f"Batch threat label shape: {threat_y.shape}")

# =========================
# MULTI-HEAD MODEL
# =========================
class SimpleCNN_MultiHead(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.shared_fc = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
        )

        self.spatial_head = nn.Linear(512, 2)
        self.threat_head = nn.Linear(512, 2)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.shared_fc(x)

        spatial_logits = self.spatial_head(x)
        threat_logits = self.threat_head(x)

        return spatial_logits, threat_logits

model = SimpleCNN_MultiHead(in_channels=in_channels).to(device)

def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

logger.info(f"Total trainable parameters: {count_parameters(model):,}")

# Freeze feature extractor initially
for p in model.features.parameters():
    p.requires_grad = False

logger.info(f"Trainable parameters head only: {count_parameters(model):,}")

# =========================
# OPTIMIZER / SCHEDULER
# =========================
optimizer = optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=config.LR_HEAD,
    weight_decay=config.WEIGHT_DECAY,
)

scheduler = None
scaler = GradScaler("cuda") if config.USE_AMP and device.type == "cuda" else None

start_epoch = 1
best_val_loss = float("inf")
epochs_without_improvement = 0

# =========================
# RESUME TRAINING
# =========================
if config.RESUME_TRAINING and config.RESUME_CHECKPOINT_PATH.exists():
    logger.info(f"Trying to resume from: {config.RESUME_CHECKPOINT_PATH}")

    try:
        ckpt = torch.load(
            config.RESUME_CHECKPOINT_PATH,
            map_location=device,
            weights_only=True
        )
    except TypeError:
        ckpt = torch.load(config.RESUME_CHECKPOINT_PATH, map_location=device)

    try:
        model.load_state_dict(ckpt["model_state"], strict=True)

        for p in model.parameters():
            p.requires_grad = True

        optimizer = optim.AdamW(
            model.parameters(),
            lr=config.LR_FULL,
            weight_decay=config.WEIGHT_DECAY,
        )

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            patience=config.SCHEDULER_PATIENCE,
            factor=config.SCHEDULER_FACTOR,
            min_lr=config.SCHEDULER_MIN_LR,
        )

        if ckpt.get("optimizer_state") is not None:
            optimizer.load_state_dict(ckpt["optimizer_state"])

        if ckpt.get("scheduler_state") is not None:
            scheduler.load_state_dict(ckpt["scheduler_state"])

        if scaler is not None and ckpt.get("scaler_state") is not None:
            scaler.load_state_dict(ckpt["scaler_state"])

        start_epoch = int(ckpt.get("epoch", 0)) + 1
        best_val_loss = float(ckpt.get("val_metrics", {}).get("loss", float("inf")))

        logger.info(f"Resumed from epoch {start_epoch}")
        logger.info(f"Previous best val loss: {best_val_loss:.4f}")

    except Exception as e:
        logger.warning(f"Could not resume because: {e}")
        logger.warning("Starting fresh stable multi-head training.")

        start_epoch = 1
        best_val_loss = float("inf")
        epochs_without_improvement = 0

else:
    logger.info("Starting training from scratch")

# =========================
# TRAIN / VALIDATE
# =========================
def train_one_epoch(model, loader):
    model.train()

    loss_sum = 0.0
    spatial_correct = 0
    threat_correct = 0
    both_correct = 0
    total = 0

    for imgs, spatial_y, threat_y in loader:
        imgs = imgs.to(device, non_blocking=True)
        spatial_y = spatial_y.to(device, non_blocking=True)
        threat_y = threat_y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if config.USE_AMP and scaler is not None:
            with autocast("cuda"):
                spatial_logits, threat_logits = model(imgs)
                loss_spatial = criterion_spatial(spatial_logits, spatial_y)
                loss_threat = criterion_threat(threat_logits, threat_y)
                loss = loss_spatial + loss_threat

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_MAX_NORM)
            scaler.step(optimizer)
            scaler.update()

        else:
            spatial_logits, threat_logits = model(imgs)
            loss_spatial = criterion_spatial(spatial_logits, spatial_y)
            loss_threat = criterion_threat(threat_logits, threat_y)
            loss = loss_spatial + loss_threat

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_MAX_NORM)
            optimizer.step()

        spatial_pred = spatial_logits.argmax(dim=1)
        threat_pred = threat_logits.argmax(dim=1)

        bs = imgs.size(0)
        loss_sum += loss.item() * bs
        spatial_correct += (spatial_pred == spatial_y).sum().item()
        threat_correct += (threat_pred == threat_y).sum().item()
        both_correct += ((spatial_pred == spatial_y) & (threat_pred == threat_y)).sum().item()
        total += bs

    return {
        "loss": loss_sum / max(total, 1),
        "spatial_acc": spatial_correct / max(total, 1),
        "threat_acc": threat_correct / max(total, 1),
        "both_acc": both_correct / max(total, 1),
    }

@torch.no_grad()
def validate(model, loader):
    model.eval()

    loss_sum = 0.0
    spatial_correct = 0
    threat_correct = 0
    both_correct = 0
    total = 0

    spatial_preds = []
    spatial_labels = []
    spatial_probs = []

    threat_preds = []
    threat_labels = []
    threat_probs = []

    for imgs, spatial_y, threat_y in loader:
        imgs = imgs.to(device, non_blocking=True)
        spatial_y = spatial_y.to(device, non_blocking=True)
        threat_y = threat_y.to(device, non_blocking=True)

        spatial_logits, threat_logits = model(imgs)

        loss_spatial = criterion_spatial(spatial_logits, spatial_y)
        loss_threat = criterion_threat(threat_logits, threat_y)
        loss = loss_spatial + loss_threat

        spatial_prob = torch.softmax(spatial_logits, dim=1)
        threat_prob = torch.softmax(threat_logits, dim=1)

        spatial_pred = spatial_prob.argmax(dim=1)
        threat_pred = threat_prob.argmax(dim=1)

        bs = imgs.size(0)
        loss_sum += loss.item() * bs
        spatial_correct += (spatial_pred == spatial_y).sum().item()
        threat_correct += (threat_pred == threat_y).sum().item()
        both_correct += ((spatial_pred == spatial_y) & (threat_pred == threat_y)).sum().item()
        total += bs

        spatial_preds.extend(spatial_pred.cpu().numpy().tolist())
        spatial_labels.extend(spatial_y.cpu().numpy().tolist())
        spatial_probs.extend(spatial_prob.cpu().numpy().tolist())

        threat_preds.extend(threat_pred.cpu().numpy().tolist())
        threat_labels.extend(threat_y.cpu().numpy().tolist())
        threat_probs.extend(threat_prob.cpu().numpy().tolist())

    def binary_metrics(y, p, probs):
        precision, recall, f1, _ = precision_recall_fscore_support(
            y, p, average="binary", zero_division=0
        )

        try:
            auc = roc_auc_score(y, np.array(probs)[:, 1])
        except Exception:
            auc = 0.0

        cm = confusion_matrix(y, p, labels=[0, 1])

        return {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "auc": float(auc),
            "cm": cm.tolist(),
        }

    spatial_m = binary_metrics(spatial_labels, spatial_preds, spatial_probs)
    threat_m = binary_metrics(threat_labels, threat_preds, threat_probs)

    return {
        "loss": loss_sum / max(total, 1),
        "spatial_acc": spatial_correct / max(total, 1),
        "threat_acc": threat_correct / max(total, 1),
        "both_acc": both_correct / max(total, 1),

        "spatial_preds": spatial_preds,
        "spatial_labels": spatial_labels,
        "spatial_probs": spatial_probs,
        "spatial_metrics": spatial_m,

        "threat_preds": threat_preds,
        "threat_labels": threat_labels,
        "threat_probs": threat_probs,
        "threat_metrics": threat_m,
    }

# =========================
# TRAIN LOOP
# =========================
logger.info("=" * 80)
logger.info("Starting STABLE MULTI-HEAD CNN training")
logger.info("Spatial head: 0=isolated, 1=overlap")
logger.info("Threat head : 0=non_contraband, 1=contraband")
logger.info("=" * 80)

for epoch in range(start_epoch, config.EPOCHS + 1):

    if start_epoch == 1 and epoch == config.UNFREEZE_EPOCH:
        logger.info("=" * 80)
        logger.info(f"Unfreezing full model at epoch {epoch}")
        logger.info("=" * 80)

        for p in model.parameters():
            p.requires_grad = True

        optimizer = optim.AdamW(
            model.parameters(),
            lr=config.LR_FULL,
            weight_decay=config.WEIGHT_DECAY,
        )

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            patience=config.SCHEDULER_PATIENCE,
            factor=config.SCHEDULER_FACTOR,
            min_lr=config.SCHEDULER_MIN_LR,
        )

        logger.info(f"Trainable parameters full model: {count_parameters(model):,}")

    train_metrics = train_one_epoch(model, train_loader)
    val_metrics = validate(model, val_loader)

    val_loss = val_metrics["loss"]

    if scheduler is not None:
        scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    checkpoint_dict = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state": scaler.state_dict() if scaler is not None else None,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "spatial_classes": config.SPATIAL_CLASSES,
        "threat_classes": config.THREAT_CLASSES,
        "seed": SEED,
    }

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        best_model_path = config.CKPT_DIR / f"{config.TRAIN_SPLIT}_best.pt"
        best_checkpoint_path = config.CKPT_DIR / f"{config.TRAIN_SPLIT}_best_checkpoint.pt"

        torch.save(model.state_dict(), best_model_path)
        torch.save(checkpoint_dict, best_checkpoint_path)

        logger.info(f"✓ New best model saved: {best_model_path}")
    else:
        epochs_without_improvement += 1

    logger.info("")
    logger.info("=" * 80)
    logger.info(f"Epoch [{epoch}/{config.EPOCHS}]")
    logger.info("=" * 80)

    logger.info(f"Train Loss:        {train_metrics['loss']:.4f}")
    logger.info(f"Train Spatial Acc: {train_metrics['spatial_acc']:.3f}")
    logger.info(f"Train Threat Acc:  {train_metrics['threat_acc']:.3f}")
    logger.info(f"Train Both Acc:    {train_metrics['both_acc']:.3f}")

    logger.info(f"Val Loss:          {val_metrics['loss']:.4f}")
    logger.info(f"Val Spatial Acc:   {val_metrics['spatial_acc']:.3f}")
    logger.info(f"Val Threat Acc:    {val_metrics['threat_acc']:.3f}")
    logger.info(f"Val Both Acc:      {val_metrics['both_acc']:.3f}")
    logger.info(f"LR:                {current_lr:.8f}")

    logger.info("Spatial metrics:")
    logger.info(f"  P={val_metrics['spatial_metrics']['precision']:.3f} "
                f"R={val_metrics['spatial_metrics']['recall']:.3f} "
                f"F1={val_metrics['spatial_metrics']['f1']:.3f} "
                f"AUC={val_metrics['spatial_metrics']['auc']:.3f}")
    logger.info(f"  CM={val_metrics['spatial_metrics']['cm']}")

    logger.info("Threat metrics:")
    logger.info(f"  P={val_metrics['threat_metrics']['precision']:.3f} "
                f"R={val_metrics['threat_metrics']['recall']:.3f} "
                f"F1={val_metrics['threat_metrics']['f1']:.3f} "
                f"AUC={val_metrics['threat_metrics']['auc']:.3f}")
    logger.info(f"  CM={val_metrics['threat_metrics']['cm']}")

    if epochs_without_improvement >= config.EARLY_STOPPING_PATIENCE:
        logger.info("=" * 80)
        logger.info(f"Early stopping at epoch {epoch}")
        logger.info(f"Best val loss: {best_val_loss:.4f}")
        logger.info("=" * 80)
        break

# =========================
# SAVE FINAL MODEL
# =========================
torch.save(model.state_dict(), config.MODEL_OUT_PATH)

metrics_path = config.MODEL_OUT_PATH.parent / "training_metrics_multihead_stable.json"
with open(metrics_path, "w") as f:
    json.dump({
        "best_val_loss": best_val_loss,
        "spatial_classes": config.SPATIAL_CLASSES,
        "threat_classes": config.THREAT_CLASSES,
    }, f, indent=2)

logger.info(f"Saved final model to: {config.MODEL_OUT_PATH}")
logger.info(f"Saved metrics to: {metrics_path}")
logger.info("Training complete")

In [ ]:
from pathlib import Path
import os, random, json, csv, logging, re
import numpy as np
from typing import Dict, List
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw

from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix

# =========================
# LOGGING
# =========================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("training_multihead_itemmask_continue.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# =========================
# CONSTANTS
# =========================
SEED = 42

GRAY_MEAN = (0.5,)
GRAY_STD = (0.25,)

MASK_MEAN = (0.5,)
MASK_STD = (0.5,)

GRADIENT_CLIP_MAX_NORM = 1.0
VALIDATION_RESIZE_FACTOR = 1.10

RANDOM_CROP_SCALE = (0.95, 1.0)
RANDOM_CROP_RATIO = (0.98, 1.02)
ROTATION_DEGREES = 2
TRANSLATE = (0.01, 0.01)
SCALE_RANGE = (0.99, 1.01)

MASK_DROPOUT_PROB = 0.10

# Only item polygons are used.
# Tray is excluded so model does not learn empty-tray shortcut.
ITEM_POLYGON_LABELS = {"shampoo", "blade"}

# =========================
# SEED
# =========================
def seed_everything(seed=42, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    logger.info(f"Set random seed to {seed}")

seed_everything(SEED)

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

# =========================
# CONFIG
# =========================
class Config:
    PROCESSED_ROOT = Path("../../data/processed/SHAMPOOBLADEINTRAY_COMPLETEV2/gray")
    INDEX_CSV = PROCESSED_ROOT / "index.csv"

    LABEL_STUDIO_JSON = Path("../../data/raw/SHAMPOOBLADEINTRAY_COMPLETEV2/result_labels.json")

    TRAIN_SPLIT = "train"
    TRAIN_LABELS_JSON = Path("../../data/labels/SHAMPOOBLADEINTRAY_COMPLETEV2/gray/train.json")
    VAL_LABELS_JSON = Path("../../data/labels/SHAMPOOBLADEINTRAY_COMPLETEV2/gray/val.json")

    MODEL_DIR = Path("../../models/classifier/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_multihead_itemmask")
    CKPT_DIR = MODEL_DIR / "checkpoints"

    # Final model from the last epoch
    MODEL_OUT_PATH = MODEL_DIR / "model.pt"
    FINAL_CHECKPOINT_PATH = CKPT_DIR / "final_checkpoint.pt"

    # Best model files.
    # These are overwritten only when new validation score is better than previous best.
    BEST_MODEL_PATH = CKPT_DIR / f"{TRAIN_SPLIT}_best.pt"
    BEST_CHECKPOINT_PATH = CKPT_DIR / f"{TRAIN_SPLIT}_best_checkpoint.pt"

    SPATIAL_CLASSES = ["isolated", "overlap"]
    THREAT_CLASSES = ["non_contraband", "contraband"]

    BATCH_SIZE = 16
    IMAGE_SIZE = 512
    EPOCHS = 50

    LR_FULL = 5e-5
    WEIGHT_DECAY = 5e-4

    SCHEDULER_PATIENCE = 5
    SCHEDULER_FACTOR = 0.7
    SCHEDULER_MIN_LR = 1e-7

    EARLY_STOPPING_PATIENCE = 10

    USE_AMP = True
    NUM_WORKERS = 2

    # =========================
    # TRAINING MODE CONTROL
    # =========================
    # False = fresh training from random weights unless PRETRAIN_FROM_PREVIOUS_BEST is True
    # True  = continue exact training from previous checkpoint, including optimizer/scheduler/scaler
    RESUME_TRAINING = False

    # True = load previous best model weights before training, but start epoch from 1
    # This is useful for improving/fine-tuning from the previous best model.
    PRETRAIN_FROM_PREVIOUS_BEST = True

    # If True, after loading previous best checkpoint, the current best_score is also loaded.
    # This means the previous best model is only overwritten if the new model becomes better.
    # Keep this True for your use case.
    KEEP_PREVIOUS_BEST_SCORE = True

    RESUME_CHECKPOINT_PATH = BEST_CHECKPOINT_PATH
    PRETRAIN_CHECKPOINT_PATH = BEST_CHECKPOINT_PATH

    @classmethod
    def create_dirs(cls):
        cls.MODEL_DIR.mkdir(parents=True, exist_ok=True)
        cls.CKPT_DIR.mkdir(parents=True, exist_ok=True)

config = Config()
config.create_dirs()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

# =========================
# BASIC HELPERS
# =========================
def norm_label(x):
    return str(x).lower().strip().replace("-", "_").replace(" ", "_")

def base_group_id(filename: str) -> str:
    """
    Convert augmented names back to original base name.

    Examples:
      abc_orig.png     -> abc
      abc_balflip1.png -> abc
      abc_balflip2.png -> abc
      abc.png          -> abc
    """
    m = re.match(r"(.+?)_(orig|balflip\d+)\.[^.]+$", filename)
    if m:
        return m.group(1)
    return Path(filename).stem

def is_balflip(filename: str) -> bool:
    return re.search(r"_balflip\d+\.[^.]+$", filename) is not None

def extract_filename_from_task(item):
    if "image" in item:
        return Path(item.get("image", "")).name

    img_ref = item.get("data", {}).get("image", "")
    return Path(img_ref).name

# =========================
# LOAD INDEX CSV
# =========================
def read_index_csv(index_csv_path: Path) -> List[Dict]:
    rows = []
    with open(index_csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        rows.extend(reader)

    if not rows:
        raise ValueError(f"index.csv is empty: {index_csv_path}")

    logger.info(f"Read {len(rows)} rows from {index_csv_path}")
    return rows

index_rows = read_index_csv(config.INDEX_CSV)

# =========================
# LOAD CLASSIFICATION LABEL JSON AS TWO HEADS
# =========================
def load_label_map(label_path: Path) -> Dict:
    data = json.load(open(label_path, "r"))

    out = {}
    bad = []

    for item in data:
        filepath = item.get("image", "").replace("\\", "/").strip()
        if not filepath:
            bad.append(("missing image", item))
            continue

        fname = Path(filepath).name

        overlap = int(item.get("overlap", 0))
        isolated = int(item.get("isolated", 0))
        contraband = int(item.get("contraband", 0))
        non_contraband = int(item.get("non_contraband", 0))

        if overlap == 1 and isolated == 0:
            spatial_label = 1
        elif overlap == 0 and isolated == 1:
            spatial_label = 0
        else:
            bad.append((filepath, f"bad spatial labels: overlap={overlap}, isolated={isolated}"))
            continue

        if contraband == 1 and non_contraband == 0:
            threat_label = 1
        elif contraband == 0 and non_contraband == 1:
            threat_label = 0
        else:
            bad.append((filepath, f"bad threat labels: contraband={contraband}, non_contraband={non_contraband}"))
            continue

        labels = {
            "spatial": spatial_label,
            "threat": threat_label,
        }

        out[filepath] = labels
        out[fname] = labels

    if bad:
        logger.warning(f"Bad label rows: {len(bad)}")
        for x in bad[:20]:
            logger.warning(x)
        raise ValueError(f"Found {len(bad)} invalid label rows")

    logger.info(f"Loaded {len(out)} label entries from {label_path}")
    return out

train_label_map = load_label_map(config.TRAIN_LABELS_JSON)
val_label_map = load_label_map(config.VAL_LABELS_JSON)

# =========================
# LOAD RAW LABEL STUDIO POLYGONS
# =========================
def load_item_polygons_from_label_studio(json_path: Path):
    """
    Loads only item polygons: Shampoo + Blade.
    Tray polygons are ignored intentionally.
    """
    data = json.load(open(json_path, "r"))

    polygons_by_filename = defaultdict(list)

    for item in data:
        if not isinstance(item, dict):
            continue

        fname = extract_filename_from_task(item)
        if not fname:
            continue

        for ann in item.get("annotations", []):
            for res in ann.get("result", []):
                if res.get("type", "").lower() not in {"polygonlabels", "polygon"}:
                    continue

                value = res.get("value", {})
                poly_labels = value.get("polygonlabels", []) or value.get("labels", [])
                points = value.get("points", [])

                if not points:
                    continue

                for lab in poly_labels:
                    lab_norm = norm_label(lab)

                    if lab_norm not in ITEM_POLYGON_LABELS:
                        continue

                    polygons_by_filename[fname].append({
                        "label": lab_norm,
                        "points": points,
                    })

    logger.info(f"Loaded item polygon masks for {len(polygons_by_filename)} original images")
    return polygons_by_filename

item_polygons_by_filename = load_item_polygons_from_label_studio(config.LABEL_STUDIO_JSON)

def find_original_polygon_key(processed_filename: str):
    """
    Finds original Label Studio filename for processed or augmented image.

    Examples:
      xxx_orig.png     -> xxx.png
      xxx_balflip1.png -> xxx.png
      xxx.png          -> xxx.png
    """
    suffix = Path(processed_filename).suffix
    original_name = base_group_id(processed_filename) + suffix

    if processed_filename in item_polygons_by_filename:
        return processed_filename

    if original_name in item_polygons_by_filename:
        return original_name

    return None

def create_item_only_mask(processed_filename: str, image_size):
    """
    Builds item-only binary mask from raw Label Studio polygons.
    Only Shampoo + Blade are drawn.

    For files named *_balflipX.png, polygon x coordinates are horizontally flipped.
    """
    w, h = image_size
    mask = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(mask)

    polygon_key = find_original_polygon_key(processed_filename)

    if polygon_key is None:
        return mask

    polygons = item_polygons_by_filename.get(polygon_key, [])
    flip_x = is_balflip(processed_filename)

    for poly in polygons:
        pts = poly["points"]

        xy = []
        for x_pct, y_pct in pts:
            if flip_x:
                x_pct = 100.0 - float(x_pct)

            x = float(x_pct) / 100.0 * w
            y = float(y_pct) / 100.0 * h
            xy.append((x, y))

        if len(xy) >= 3:
            draw.polygon(xy, fill=255)

    return mask

# =========================
# JOINT TRANSFORMS FOR IMAGE + ITEM MASK
# =========================
class JointTrainTransform:
    def __init__(self, image_size):
        self.image_size = image_size

    def __call__(self, img, item_mask):
        i, j, h, w = T.RandomResizedCrop.get_params(
            img,
            scale=RANDOM_CROP_SCALE,
            ratio=RANDOM_CROP_RATIO,
        )

        img = TF.resized_crop(
            img, i, j, h, w,
            size=[self.image_size, self.image_size],
            interpolation=TF.InterpolationMode.BILINEAR,
        )

        item_mask = TF.resized_crop(
            item_mask, i, j, h, w,
            size=[self.image_size, self.image_size],
            interpolation=TF.InterpolationMode.NEAREST,
        )

        angle = random.uniform(-ROTATION_DEGREES, ROTATION_DEGREES)

        max_dx = TRANSLATE[0] * self.image_size
        max_dy = TRANSLATE[1] * self.image_size

        translations = (
            int(round(random.uniform(-max_dx, max_dx))),
            int(round(random.uniform(-max_dy, max_dy))),
        )

        scale = random.uniform(SCALE_RANGE[0], SCALE_RANGE[1])
        shear = [0.0, 0.0]

        img = TF.affine(
            img,
            angle=angle,
            translate=translations,
            scale=scale,
            shear=shear,
            interpolation=TF.InterpolationMode.BILINEAR,
            fill=0,
        )

        item_mask = TF.affine(
            item_mask,
            angle=angle,
            translate=translations,
            scale=scale,
            shear=shear,
            interpolation=TF.InterpolationMode.NEAREST,
            fill=0,
        )

        img_t = TF.to_tensor(img)
        mask_t = TF.to_tensor(item_mask)

        img_t = TF.normalize(img_t, GRAY_MEAN, GRAY_STD)
        mask_t = TF.normalize(mask_t, MASK_MEAN, MASK_STD)

        if random.random() < MASK_DROPOUT_PROB:
            mask_t = torch.zeros_like(mask_t)

        x = torch.cat([img_t, mask_t], dim=0)
        return x


class JointValTransform:
    def __init__(self, image_size):
        self.image_size = image_size
        self.resize_size = int(image_size * VALIDATION_RESIZE_FACTOR)

    def __call__(self, img, item_mask):
        img = TF.resize(
            img,
            [self.resize_size],
            interpolation=TF.InterpolationMode.BILINEAR,
        )

        item_mask = TF.resize(
            item_mask,
            [self.resize_size],
            interpolation=TF.InterpolationMode.NEAREST,
        )

        img = TF.center_crop(img, [self.image_size, self.image_size])
        item_mask = TF.center_crop(item_mask, [self.image_size, self.image_size])

        img_t = TF.to_tensor(img)
        mask_t = TF.to_tensor(item_mask)

        img_t = TF.normalize(img_t, GRAY_MEAN, GRAY_STD)
        mask_t = TF.normalize(mask_t, MASK_MEAN, MASK_STD)

        x = torch.cat([img_t, mask_t], dim=0)
        return x

# =========================
# DATASET
# =========================
class ProcessedSplitDataset_ItemMask(Dataset):
    def __init__(self, index_rows, processed_root, split, label_map, transform=None):
        self.processed_root = processed_root
        self.split = split.lower()
        self.transform = transform

        self.filepaths = []
        self.spatial_labels = []
        self.threat_labels = []
        self.has_item_polygon = []

        missing = []

        for r in index_rows:
            fp = (r.get("filepath") or "").replace("\\", "/").strip()
            row_split = (r.get("split") or "").strip().lower()

            if row_split != self.split:
                continue

            fname = Path(fp).name
            label = label_map.get(fp, label_map.get(fname, None))

            if label is None:
                missing.append(fp)
                continue

            img_path = self.processed_root / fp

            if not img_path.exists():
                missing.append(str(img_path))
                continue

            polygon_key = find_original_polygon_key(fname)

            self.filepaths.append(fp)
            self.spatial_labels.append(int(label["spatial"]))
            self.threat_labels.append(int(label["threat"]))
            self.has_item_polygon.append(polygon_key is not None)

        if missing:
            raise KeyError(f"Missing labels/files: {len(missing)} examples. First few: {missing[:10]}")

        logger.info(f"Dataset {split}: {len(self.filepaths)} samples loaded")
        logger.info(f"Dataset {split}: {sum(self.has_item_polygon)} samples have item polygons")
        logger.info(f"Dataset {split}: {len(self.has_item_polygon) - sum(self.has_item_polygon)} samples have blank item mask")

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img_path = self.processed_root / self.filepaths[idx]

        img = Image.open(img_path).convert("L")

        item_mask = create_item_only_mask(
            processed_filename=Path(self.filepaths[idx]).name,
            image_size=img.size,
        )

        if self.transform:
            x = self.transform(img, item_mask)
        else:
            img_t = TF.normalize(TF.to_tensor(img), GRAY_MEAN, GRAY_STD)
            mask_t = TF.normalize(TF.to_tensor(item_mask), MASK_MEAN, MASK_STD)
            x = torch.cat([img_t, mask_t], dim=0)

        spatial_y = torch.tensor(self.spatial_labels[idx], dtype=torch.long)
        threat_y = torch.tensor(self.threat_labels[idx], dtype=torch.long)

        return x, spatial_y, threat_y

# =========================
# CREATE DATASETS
# =========================
train_transform = JointTrainTransform(config.IMAGE_SIZE)
val_transform = JointValTransform(config.IMAGE_SIZE)

train_ds = ProcessedSplitDataset_ItemMask(
    index_rows=index_rows,
    processed_root=config.PROCESSED_ROOT,
    split=config.TRAIN_SPLIT,
    label_map=train_label_map,
    transform=train_transform,
)

val_ds = ProcessedSplitDataset_ItemMask(
    index_rows=index_rows,
    processed_root=config.PROCESSED_ROOT,
    split="val",
    label_map=val_label_map,
    transform=val_transform,
)

logger.info(f"Train spatial distribution: {Counter(train_ds.spatial_labels)}")
logger.info(f"Train threat distribution:  {Counter(train_ds.threat_labels)}")
logger.info(f"Val spatial distribution:   {Counter(val_ds.spatial_labels)}")
logger.info(f"Val threat distribution:    {Counter(val_ds.threat_labels)}")

# =========================
# CLASS WEIGHTS
# =========================
def make_class_weights(labels, num_classes=2):
    counts = np.bincount(labels, minlength=num_classes)
    counts = np.maximum(counts, 1)

    weights = counts.sum() / (num_classes * counts)
    weights = np.sqrt(weights)
    weights = np.clip(weights, 0.85, 1.25)

    return torch.tensor(weights, dtype=torch.float32).to(device)

spatial_weights = make_class_weights(train_ds.spatial_labels, 2)
threat_weights = make_class_weights(train_ds.threat_labels, 2)

logger.info(f"spatial_weights: {spatial_weights.tolist()}")
logger.info(f"threat_weights: {threat_weights.tolist()}")

criterion_spatial = nn.CrossEntropyLoss(weight=spatial_weights, label_smoothing=0.03)
criterion_threat = nn.CrossEntropyLoss(weight=threat_weights, label_smoothing=0.03)

# =========================
# LOADERS
# =========================
train_loader = DataLoader(
    train_ds,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)

x, spatial_y, threat_y = next(iter(train_loader))
logger.info(f"Batch input shape: {x.shape}")  # should be [B, 2, 512, 512]
logger.info(f"Batch spatial label shape: {spatial_y.shape}")
logger.info(f"Batch threat label shape: {threat_y.shape}")

# =========================
# MULTI-HEAD MODEL
# =========================
class SimpleCNN_MultiHead(nn.Module):
    def __init__(self, in_channels=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.shared_fc = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
        )

        self.spatial_head = nn.Linear(512, 2)
        self.threat_head = nn.Linear(512, 2)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.shared_fc(x)

        spatial_logits = self.spatial_head(x)
        threat_logits = self.threat_head(x)

        return spatial_logits, threat_logits

model = SimpleCNN_MultiHead(in_channels=2).to(device)

def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

logger.info(f"Total trainable parameters: {count_parameters(model):,}")

# =========================
# OPTIMIZER / SCHEDULER / SCALER
# =========================
optimizer = optim.AdamW(
    model.parameters(),
    lr=config.LR_FULL,
    weight_decay=config.WEIGHT_DECAY,
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    patience=config.SCHEDULER_PATIENCE,
    factor=config.SCHEDULER_FACTOR,
    min_lr=config.SCHEDULER_MIN_LR,
)

scaler = GradScaler("cuda") if config.USE_AMP and device.type == "cuda" else None

start_epoch = 1
best_score = -float("inf")
best_val_loss = float("inf")
epochs_without_improvement = 0

# =========================
# CHECKPOINT LOADING
# =========================
def safe_load_checkpoint(path: Path):
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)

def load_model_state_from_checkpoint(model, ckpt):
    if isinstance(ckpt, dict):
        if "model_state" in ckpt:
            model.load_state_dict(ckpt["model_state"], strict=True)
        elif "model_state_dict" in ckpt:
            model.load_state_dict(ckpt["model_state_dict"], strict=True)
        else:
            model.load_state_dict(ckpt, strict=True)
    else:
        raise ValueError("Checkpoint format is not supported.")

# =========================
# RESUME OR PRETRAIN
# =========================
if config.RESUME_TRAINING and config.RESUME_CHECKPOINT_PATH.exists():
    logger.info("=" * 80)
    logger.info(f"RESUME_TRAINING=True")
    logger.info(f"Continuing full training from: {config.RESUME_CHECKPOINT_PATH}")
    logger.info("=" * 80)

    ckpt = safe_load_checkpoint(config.RESUME_CHECKPOINT_PATH)

    try:
        load_model_state_from_checkpoint(model, ckpt)

        if isinstance(ckpt, dict):
            if ckpt.get("optimizer_state") is not None:
                optimizer.load_state_dict(ckpt["optimizer_state"])

            if ckpt.get("scheduler_state") is not None:
                scheduler.load_state_dict(ckpt["scheduler_state"])

            if scaler is not None and ckpt.get("scaler_state") is not None:
                scaler.load_state_dict(ckpt["scaler_state"])

            start_epoch = int(ckpt.get("epoch", 0)) + 1
            best_score = float(ckpt.get("best_score", -float("inf")))
            best_val_loss = float(ckpt.get("best_val_loss", float("inf")))

        logger.info(f"Resumed from epoch {start_epoch}")
        logger.info(f"Previous best score: {best_score:.4f}")
        logger.info(f"Previous best val loss: {best_val_loss:.4f}")

    except Exception as e:
        logger.warning(f"Could not resume because: {e}")
        logger.warning("Starting fresh training instead.")
        start_epoch = 1
        best_score = -float("inf")
        best_val_loss = float("inf")
        epochs_without_improvement = 0

elif config.PRETRAIN_FROM_PREVIOUS_BEST and config.PRETRAIN_CHECKPOINT_PATH.exists():
    logger.info("=" * 80)
    logger.info(f"PRETRAIN_FROM_PREVIOUS_BEST=True")
    logger.info(f"Loading model weights from previous best: {config.PRETRAIN_CHECKPOINT_PATH}")
    logger.info("Optimizer/scheduler/scaler are reset.")
    logger.info("Training starts again from epoch 1.")
    logger.info("=" * 80)

    ckpt = safe_load_checkpoint(config.PRETRAIN_CHECKPOINT_PATH)

    try:
        load_model_state_from_checkpoint(model, ckpt)

        start_epoch = 1

        if config.KEEP_PREVIOUS_BEST_SCORE and isinstance(ckpt, dict):
            best_score = float(ckpt.get("best_score", -float("inf")))
            best_val_loss = float(ckpt.get("best_val_loss", float("inf")))

            logger.info(f"Loaded previous best score: {best_score:.4f}")
            logger.info(f"Loaded previous best val loss: {best_val_loss:.4f}")
            logger.info("Previous best checkpoint will only be overwritten if new score is better.")
        else:
            best_score = -float("inf")
            best_val_loss = float("inf")
            logger.info("Previous best score was not kept. First improved run can overwrite best checkpoint.")

    except Exception as e:
        logger.warning(f"Could not pretrain from previous best because: {e}")
        logger.warning("Starting fresh training instead.")
        start_epoch = 1
        best_score = -float("inf")
        best_val_loss = float("inf")
        epochs_without_improvement = 0

else:
    logger.info("=" * 80)
    logger.info("Starting training from scratch")
    logger.info("=" * 80)

# =========================
# TRAIN / VALIDATE
# =========================
def train_one_epoch(model, loader):
    model.train()

    loss_sum = 0.0
    spatial_correct = 0
    threat_correct = 0
    both_correct = 0
    total = 0

    for imgs, spatial_y, threat_y in loader:
        imgs = imgs.to(device, non_blocking=True)
        spatial_y = spatial_y.to(device, non_blocking=True)
        threat_y = threat_y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if config.USE_AMP and scaler is not None:
            with autocast("cuda"):
                spatial_logits, threat_logits = model(imgs)
                loss_spatial = criterion_spatial(spatial_logits, spatial_y)
                loss_threat = criterion_threat(threat_logits, threat_y)
                loss = loss_spatial + loss_threat

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_MAX_NORM)
            scaler.step(optimizer)
            scaler.update()

        else:
            spatial_logits, threat_logits = model(imgs)
            loss_spatial = criterion_spatial(spatial_logits, spatial_y)
            loss_threat = criterion_threat(threat_logits, threat_y)
            loss = loss_spatial + loss_threat

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_MAX_NORM)
            optimizer.step()

        spatial_pred = spatial_logits.argmax(dim=1)
        threat_pred = threat_logits.argmax(dim=1)

        bs = imgs.size(0)
        loss_sum += loss.item() * bs
        spatial_correct += (spatial_pred == spatial_y).sum().item()
        threat_correct += (threat_pred == threat_y).sum().item()
        both_correct += ((spatial_pred == spatial_y) & (threat_pred == threat_y)).sum().item()
        total += bs

    return {
        "loss": loss_sum / max(total, 1),
        "spatial_acc": spatial_correct / max(total, 1),
        "threat_acc": threat_correct / max(total, 1),
        "both_acc": both_correct / max(total, 1),
    }


@torch.no_grad()
def validate(model, loader):
    model.eval()

    loss_sum = 0.0
    spatial_correct = 0
    threat_correct = 0
    both_correct = 0
    total = 0

    spatial_preds = []
    spatial_labels = []
    spatial_probs = []

    threat_preds = []
    threat_labels = []
    threat_probs = []

    for imgs, spatial_y, threat_y in loader:
        imgs = imgs.to(device, non_blocking=True)
        spatial_y = spatial_y.to(device, non_blocking=True)
        threat_y = threat_y.to(device, non_blocking=True)

        spatial_logits, threat_logits = model(imgs)

        loss_spatial = criterion_spatial(spatial_logits, spatial_y)
        loss_threat = criterion_threat(threat_logits, threat_y)
        loss = loss_spatial + loss_threat

        spatial_prob = torch.softmax(spatial_logits, dim=1)
        threat_prob = torch.softmax(threat_logits, dim=1)

        spatial_pred = spatial_prob.argmax(dim=1)
        threat_pred = threat_prob.argmax(dim=1)

        bs = imgs.size(0)
        loss_sum += loss.item() * bs
        spatial_correct += (spatial_pred == spatial_y).sum().item()
        threat_correct += (threat_pred == threat_y).sum().item()
        both_correct += ((spatial_pred == spatial_y) & (threat_pred == threat_y)).sum().item()
        total += bs

        spatial_preds.extend(spatial_pred.cpu().numpy().tolist())
        spatial_labels.extend(spatial_y.cpu().numpy().tolist())
        spatial_probs.extend(spatial_prob.cpu().numpy().tolist())

        threat_preds.extend(threat_pred.cpu().numpy().tolist())
        threat_labels.extend(threat_y.cpu().numpy().tolist())
        threat_probs.extend(threat_prob.cpu().numpy().tolist())

    def binary_metrics(y, p, probs):
        precision, recall, f1, _ = precision_recall_fscore_support(
            y, p, average="binary", zero_division=0
        )

        try:
            auc = roc_auc_score(y, np.array(probs)[:, 1])
        except Exception:
            auc = 0.0

        cm = confusion_matrix(y, p, labels=[0, 1])

        return {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "auc": float(auc),
            "cm": cm.tolist(),
        }

    spatial_m = binary_metrics(spatial_labels, spatial_preds, spatial_probs)
    threat_m = binary_metrics(threat_labels, threat_preds, threat_probs)

    return {
        "loss": loss_sum / max(total, 1),
        "spatial_acc": spatial_correct / max(total, 1),
        "threat_acc": threat_correct / max(total, 1),
        "both_acc": both_correct / max(total, 1),
        "spatial_metrics": spatial_m,
        "threat_metrics": threat_m,
    }


def compute_selection_score(val_metrics):
    both_acc = val_metrics["both_acc"]
    spatial_auc = val_metrics["spatial_metrics"]["auc"]
    threat_auc = val_metrics["threat_metrics"]["auc"]
    spatial_f1 = val_metrics["spatial_metrics"]["f1"]
    threat_f1 = val_metrics["threat_metrics"]["f1"]

    score = (
        0.40 * both_acc +
        0.20 * spatial_auc +
        0.20 * threat_auc +
        0.10 * spatial_f1 +
        0.10 * threat_f1
    )

    return float(score)

# =========================
# TRAIN LOOP
# =========================
logger.info("=" * 80)
logger.info("Starting ITEM-ONLY MASK MULTI-HEAD CNN training")
logger.info("Input channels: 0=grayscale image, 1=item-only mask")
logger.info("Item mask labels: Shampoo + Blade only")
logger.info("Tray polygon is excluded from mask channel")
logger.info("Spatial head: 0=isolated, 1=overlap")
logger.info("Threat head : 0=non_contraband, 1=contraband")
logger.info(f"Image size: {config.IMAGE_SIZE}")
logger.info(f"Batch size: {config.BATCH_SIZE}")
logger.info(f"LR: {config.LR_FULL}")
logger.info(f"Mask dropout prob: {MASK_DROPOUT_PROB}")
logger.info(f"Start epoch: {start_epoch}")
logger.info(f"Current best score to beat: {best_score:.4f}")
logger.info("=" * 80)

for epoch in range(start_epoch, config.EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader)
    val_metrics = validate(model, val_loader)

    val_loss = val_metrics["loss"]
    selection_score = compute_selection_score(val_metrics)

    scheduler.step(selection_score)

    current_lr = optimizer.param_groups[0]["lr"]

    if val_loss < best_val_loss:
        best_val_loss = val_loss

    checkpoint_dict = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict() if scaler is not None else None,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "best_score": max(best_score, selection_score),
        "best_val_loss": best_val_loss,
        "selection_score": selection_score,
        "spatial_classes": config.SPATIAL_CLASSES,
        "threat_classes": config.THREAT_CLASSES,
        "input_channels": ["grayscale_image", "item_only_mask"],
        "item_mask_labels": ["shampoo", "blade"],
        "excluded_mask_labels": ["tray"],
        "seed": SEED,
        "image_size": config.IMAGE_SIZE,
        "gray_mean": GRAY_MEAN,
        "gray_std": GRAY_STD,
        "mask_mean": MASK_MEAN,
        "mask_std": MASK_STD,
        "mask_dropout_prob": MASK_DROPOUT_PROB,
        "pretrained_from_previous_best": bool(
            config.PRETRAIN_FROM_PREVIOUS_BEST and config.PRETRAIN_CHECKPOINT_PATH.exists()
        ),
    }

    if selection_score > best_score:
        best_score = selection_score
        epochs_without_improvement = 0

        # Overwrite previous best files
        torch.save(model.state_dict(), config.BEST_MODEL_PATH)
        torch.save(checkpoint_dict, config.BEST_CHECKPOINT_PATH)

        logger.info(f"✓ New best model saved and previous best overwritten: {config.BEST_MODEL_PATH}")
        logger.info(f"✓ New best checkpoint saved and previous best overwritten: {config.BEST_CHECKPOINT_PATH}")
        logger.info(f"✓ New best score: {best_score:.4f}")
    else:
        epochs_without_improvement += 1

    logger.info("")
    logger.info("=" * 80)
    logger.info(f"Epoch [{epoch}/{config.EPOCHS}]")
    logger.info("=" * 80)

    logger.info(f"Train Loss:        {train_metrics['loss']:.4f}")
    logger.info(f"Train Spatial Acc: {train_metrics['spatial_acc']:.3f}")
    logger.info(f"Train Threat Acc:  {train_metrics['threat_acc']:.3f}")
    logger.info(f"Train Both Acc:    {train_metrics['both_acc']:.3f}")

    logger.info(f"Val Loss:          {val_metrics['loss']:.4f}")
    logger.info(f"Val Spatial Acc:   {val_metrics['spatial_acc']:.3f}")
    logger.info(f"Val Threat Acc:    {val_metrics['threat_acc']:.3f}")
    logger.info(f"Val Both Acc:      {val_metrics['both_acc']:.3f}")
    logger.info(f"Selection Score:   {selection_score:.4f}")
    logger.info(f"Best Score:        {best_score:.4f}")
    logger.info(f"Best Val Loss:     {best_val_loss:.4f}")
    logger.info(f"Epochs no improve: {epochs_without_improvement}")
    logger.info(f"LR:                {current_lr:.8f}")

    logger.info("Spatial metrics:")
    logger.info(f"  P={val_metrics['spatial_metrics']['precision']:.3f} "
                f"R={val_metrics['spatial_metrics']['recall']:.3f} "
                f"F1={val_metrics['spatial_metrics']['f1']:.3f} "
                f"AUC={val_metrics['spatial_metrics']['auc']:.3f}")
    logger.info(f"  CM={val_metrics['spatial_metrics']['cm']}")

    logger.info("Threat metrics:")
    logger.info(f"  P={val_metrics['threat_metrics']['precision']:.3f} "
                f"R={val_metrics['threat_metrics']['recall']:.3f} "
                f"F1={val_metrics['threat_metrics']['f1']:.3f} "
                f"AUC={val_metrics['threat_metrics']['auc']:.3f}")
    logger.info(f"  CM={val_metrics['threat_metrics']['cm']}")

    if epochs_without_improvement >= config.EARLY_STOPPING_PATIENCE:
        logger.info("=" * 80)
        logger.info(f"Early stopping at epoch {epoch}")
        logger.info(f"Best selection score: {best_score:.4f}")
        logger.info(f"Best val loss: {best_val_loss:.4f}")
        logger.info("=" * 80)
        break

# =========================
# SAVE FINAL MODEL
# =========================
torch.save(model.state_dict(), config.MODEL_OUT_PATH)

final_checkpoint = {
    "epoch": epoch,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "scheduler_state": scheduler.state_dict(),
    "scaler_state": scaler.state_dict() if scaler is not None else None,
    "best_score": best_score,
    "best_val_loss": best_val_loss,
    "spatial_classes": config.SPATIAL_CLASSES,
    "threat_classes": config.THREAT_CLASSES,
    "input_channels": ["grayscale_image", "item_only_mask"],
    "item_mask_labels": ["shampoo", "blade"],
    "excluded_mask_labels": ["tray"],
    "image_size": config.IMAGE_SIZE,
    "gray_mean": GRAY_MEAN,
    "gray_std": GRAY_STD,
    "mask_mean": MASK_MEAN,
    "mask_std": MASK_STD,
    "mask_dropout_prob": MASK_DROPOUT_PROB,
    "lr_full": config.LR_FULL,
    "weight_decay": config.WEIGHT_DECAY,
    "seed": SEED,
}

torch.save(final_checkpoint, config.FINAL_CHECKPOINT_PATH)

metrics_path = config.MODEL_DIR / "training_metrics_itemmask.json"

with open(metrics_path, "w") as f:
    json.dump({
        "best_score": best_score,
        "best_val_loss": best_val_loss,
        "spatial_classes": config.SPATIAL_CLASSES,
        "threat_classes": config.THREAT_CLASSES,
        "input_channels": ["grayscale_image", "item_only_mask"],
        "item_mask_labels": ["shampoo", "blade"],
        "excluded_mask_labels": ["tray"],
        "image_size": config.IMAGE_SIZE,
        "gray_mean": GRAY_MEAN,
        "gray_std": GRAY_STD,
        "mask_mean": MASK_MEAN,
        "mask_std": MASK_STD,
        "mask_dropout_prob": MASK_DROPOUT_PROB,
        "lr_full": config.LR_FULL,
        "weight_decay": config.WEIGHT_DECAY,
        "seed": SEED,
        "best_model_path": str(config.BEST_MODEL_PATH),
        "best_checkpoint_path": str(config.BEST_CHECKPOINT_PATH),
        "final_model_path": str(config.MODEL_OUT_PATH),
        "final_checkpoint_path": str(config.FINAL_CHECKPOINT_PATH),
    }, f, indent=2)

logger.info(f"Saved final model to: {config.MODEL_OUT_PATH}")
logger.info(f"Saved final checkpoint to: {config.FINAL_CHECKPOINT_PATH}")
logger.info(f"Saved metrics to: {metrics_path}")
logger.info(f"Best model path: {config.BEST_MODEL_PATH}")
logger.info(f"Best checkpoint path: {config.BEST_CHECKPOINT_PATH}")
logger.info("Training complete")

Train the model progressively with lesser mask

In [ ]:
from pathlib import Path
import os, random, json, csv, logging, re
import numpy as np
from typing import Dict, List
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw

from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix

# =========================
# LOGGING
# =========================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("training_multihead_itemmask_optional_slow.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# =========================
# CONSTANTS
# =========================
SEED = 42

GRAY_MEAN = (0.5,)
GRAY_STD = (0.25,)

MASK_MEAN = (0.5,)
MASK_STD = (0.5,)

# Important:
# Blank mask before normalization = pixel 0.
# After normalize with mean=0.5, std=0.5:
# (0 - 0.5) / 0.5 = -1.0
BLANK_MASK_VALUE_NORM = -1.0

GRADIENT_CLIP_MAX_NORM = 1.0
VALIDATION_RESIZE_FACTOR = 1.10

RANDOM_CROP_SCALE = (0.95, 1.0)
RANDOM_CROP_RATIO = (0.98, 1.02)
ROTATION_DEGREES = 2
TRANSLATE = (0.01, 0.01)
SCALE_RANGE = (0.99, 1.01)

ITEM_POLYGON_LABELS = {"shampoo", "blade"}

# =========================
# SEED
# =========================
def seed_everything(seed=42, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    logger.info(f"Set random seed to {seed}")

seed_everything(SEED)

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

# =========================
# CONFIG
# =========================
class Config:
    PROCESSED_ROOT = Path("../../data/processed/SHAMPOOBLADEINTRAY_COMPLETEV2/gray")
    INDEX_CSV = PROCESSED_ROOT / "index.csv"

    LABEL_STUDIO_JSON = Path("../../data/raw/SHAMPOOBLADEINTRAY_COMPLETEV2/result_labels.json")

    TRAIN_SPLIT = "train"
    TRAIN_LABELS_JSON = Path("../../data/labels/SHAMPOOBLADEINTRAY_COMPLETEV2/gray/train.json")
    VAL_LABELS_JSON = Path("../../data/labels/SHAMPOOBLADEINTRAY_COMPLETEV2/gray/val.json")

    # Save to a NEW folder so your old good item-mask model is not overwritten
    MODEL_DIR = Path("../../models/classifier/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_multihead_itemmask_optional_slow")
    CKPT_DIR = MODEL_DIR / "checkpoints"

    MODEL_OUT_PATH = MODEL_DIR / "model.pt"
    FINAL_CHECKPOINT_PATH = CKPT_DIR / "final_checkpoint.pt"

    BEST_MODEL_PATH = CKPT_DIR / f"{TRAIN_SPLIT}_best.pt"
    BEST_CHECKPOINT_PATH = CKPT_DIR / f"{TRAIN_SPLIT}_best_checkpoint.pt"

    # Previous strong item-mask checkpoint to initialize from
    SOURCE_PRETRAIN_CHECKPOINT_PATH = Path(
        "../../models/classifier/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_multihead_itemmask/checkpoints/train_best_checkpoint.pt"
    )

    SPATIAL_CLASSES = ["isolated", "overlap"]
    THREAT_CLASSES = ["non_contraband", "contraband"]

    BATCH_SIZE = 16
    IMAGE_SIZE = 512
    EPOCHS = 100

    # Lower LR because we are fine-tuning slowly
    LR_FULL = 2e-5
    WEIGHT_DECAY = 5e-4

    SCHEDULER_PATIENCE = 6
    SCHEDULER_FACTOR = 0.7
    SCHEDULER_MIN_LR = 1e-7

    EARLY_STOPPING_PATIENCE = 150

    USE_AMP = True
    NUM_WORKERS = 2

    RESUME_TRAINING = False

    # Use previous model weights but train a new optional-mask model
    PRETRAIN_FROM_PREVIOUS_BEST = True

    # Because this is a new objective, do NOT keep old score.
    # The old best was measured mostly with masks.
    KEEP_PREVIOUS_BEST_SCORE = False

    RESUME_CHECKPOINT_PATH = BEST_CHECKPOINT_PATH
    PRETRAIN_CHECKPOINT_PATH = SOURCE_PRETRAIN_CHECKPOINT_PATH

    # Slightly penalize missing contraband more, but not too extreme
    THREAT_CONTRABAND_WEIGHT_MULT = 1.25

    @classmethod
    def create_dirs(cls):
        cls.MODEL_DIR.mkdir(parents=True, exist_ok=True)
        cls.CKPT_DIR.mkdir(parents=True, exist_ok=True)

config = Config()
config.create_dirs()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

# =========================
# PROGRESSIVE SCHEDULE
# =========================
def get_progressive_mask_dropout(epoch: int) -> float:
    """
    Slowly increases mask dropout.
    Early training: model can use the mask.
    Later training: model must learn image-only features too.
    """
    if epoch <= 5:
        return 0.15
    elif epoch <= 15:
        return 0.30
    elif epoch <= 30:
        return 0.45
    else:
        return 0.60


def get_blank_consistency_weight(epoch: int) -> float:
    """
    Trains each batch twice:
      1. image + item mask
      2. image + blank mask

    This weight slowly increases so the model does not suddenly collapse.
    """
    if epoch <= 5:
        return 0.10
    elif epoch <= 15:
        return 0.30
    elif epoch <= 30:
        return 0.50
    else:
        return 0.70


# =========================
# BASIC HELPERS
# =========================
def norm_label(x):
    return str(x).lower().strip().replace("-", "_").replace(" ", "_")

def base_group_id(filename: str) -> str:
    m = re.match(r"(.+?)_(orig|balflip\d+)\.[^.]+$", filename)
    if m:
        return m.group(1)
    return Path(filename).stem

def is_balflip(filename: str) -> bool:
    return re.search(r"_balflip\d+\.[^.]+$", filename) is not None

def extract_filename_from_task(item):
    if "image" in item:
        return Path(item.get("image", "")).name

    img_ref = item.get("data", {}).get("image", "")
    return Path(img_ref).name

# =========================
# LOAD INDEX CSV
# =========================
def read_index_csv(index_csv_path: Path) -> List[Dict]:
    rows = []
    with open(index_csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        rows.extend(reader)

    if not rows:
        raise ValueError(f"index.csv is empty: {index_csv_path}")

    logger.info(f"Read {len(rows)} rows from {index_csv_path}")
    return rows

index_rows = read_index_csv(config.INDEX_CSV)

# =========================
# LOAD LABELS
# =========================
def load_label_map(label_path: Path) -> Dict:
    data = json.load(open(label_path, "r"))

    out = {}
    bad = []

    for item in data:
        filepath = item.get("image", "").replace("\\", "/").strip()
        if not filepath:
            bad.append(("missing image", item))
            continue

        fname = Path(filepath).name

        overlap = int(item.get("overlap", 0))
        isolated = int(item.get("isolated", 0))
        contraband = int(item.get("contraband", 0))
        non_contraband = int(item.get("non_contraband", 0))

        if overlap == 1 and isolated == 0:
            spatial_label = 1
        elif overlap == 0 and isolated == 1:
            spatial_label = 0
        else:
            bad.append((filepath, f"bad spatial labels: overlap={overlap}, isolated={isolated}"))
            continue

        if contraband == 1 and non_contraband == 0:
            threat_label = 1
        elif contraband == 0 and non_contraband == 1:
            threat_label = 0
        else:
            bad.append((filepath, f"bad threat labels: contraband={contraband}, non_contraband={non_contraband}"))
            continue

        labels = {
            "spatial": spatial_label,
            "threat": threat_label,
        }

        out[filepath] = labels
        out[fname] = labels

    if bad:
        logger.warning(f"Bad label rows: {len(bad)}")
        for x in bad[:20]:
            logger.warning(x)
        raise ValueError(f"Found {len(bad)} invalid label rows")

    logger.info(f"Loaded {len(out)} label entries from {label_path}")
    return out

train_label_map = load_label_map(config.TRAIN_LABELS_JSON)
val_label_map = load_label_map(config.VAL_LABELS_JSON)

# =========================
# LOAD ITEM POLYGONS
# =========================
def load_item_polygons_from_label_studio(json_path: Path):
    data = json.load(open(json_path, "r"))
    polygons_by_filename = defaultdict(list)

    for item in data:
        if not isinstance(item, dict):
            continue

        fname = extract_filename_from_task(item)
        if not fname:
            continue

        for ann in item.get("annotations", []):
            for res in ann.get("result", []):
                if res.get("type", "").lower() not in {"polygonlabels", "polygon"}:
                    continue

                value = res.get("value", {})
                poly_labels = value.get("polygonlabels", []) or value.get("labels", [])
                points = value.get("points", [])

                if not points:
                    continue

                for lab in poly_labels:
                    lab_norm = norm_label(lab)

                    if lab_norm not in ITEM_POLYGON_LABELS:
                        continue

                    polygons_by_filename[fname].append({
                        "label": lab_norm,
                        "points": points,
                    })

    logger.info(f"Loaded item polygon masks for {len(polygons_by_filename)} original images")
    return polygons_by_filename

item_polygons_by_filename = load_item_polygons_from_label_studio(config.LABEL_STUDIO_JSON)

def find_original_polygon_key(processed_filename: str):
    suffix = Path(processed_filename).suffix
    original_name = base_group_id(processed_filename) + suffix

    if processed_filename in item_polygons_by_filename:
        return processed_filename

    if original_name in item_polygons_by_filename:
        return original_name

    return None

def create_item_only_mask(processed_filename: str, image_size):
    w, h = image_size
    mask = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(mask)

    polygon_key = find_original_polygon_key(processed_filename)

    if polygon_key is None:
        return mask

    polygons = item_polygons_by_filename.get(polygon_key, [])
    flip_x = is_balflip(processed_filename)

    for poly in polygons:
        pts = poly["points"]

        xy = []
        for x_pct, y_pct in pts:
            if flip_x:
                x_pct = 100.0 - float(x_pct)

            x = float(x_pct) / 100.0 * w
            y = float(y_pct) / 100.0 * h
            xy.append((x, y))

        if len(xy) >= 3:
            draw.polygon(xy, fill=255)

    return mask

# =========================
# TRANSFORMS — KEEP ASPECT RATIO, NO CROP
# =========================
def resize_keep_ratio_and_pad(pil_img, target_size, interpolation, fill=255):
    """
    Resize image to fit inside target_size x target_size without cropping,
    then pad to target_size x target_size.

    For X-ray image:
      fill=255 means white background padding.

    For mask:
      fill=0 means black/empty mask padding.
    """
    img = pil_img.convert("L")
    w, h = img.size

    scale = min(target_size / w, target_size / h)

    new_w = int(round(w * scale))
    new_h = int(round(h * scale))

    img = TF.resize(
        img,
        [new_h, new_w],
        interpolation=interpolation,
    )

    pad_left = (target_size - new_w) // 2
    pad_top = (target_size - new_h) // 2
    pad_right = target_size - new_w - pad_left
    pad_bottom = target_size - new_h - pad_top

    img = TF.pad(
        img,
        padding=[pad_left, pad_top, pad_right, pad_bottom],
        fill=fill,
    )

    return img


class JointTrainTransform:
    """
    Training transform:
    - Keeps full image visible
    - Keeps aspect ratio
    - Pads to 512x512
    - Does NOT crop tray/item
    - Still applies tiny affine augmentation after padding
    """

    def __init__(self, image_size, mask_dropout_prob=0.15):
        self.image_size = image_size
        self.mask_dropout_prob = mask_dropout_prob

    def set_mask_dropout_prob(self, p: float):
        self.mask_dropout_prob = float(p)

    def __call__(self, img, item_mask):
        # 1. Resize with aspect ratio and pad to 512x512
        img = resize_keep_ratio_and_pad(
            img,
            target_size=self.image_size,
            interpolation=TF.InterpolationMode.BILINEAR,
            fill=255,
        )

        item_mask = resize_keep_ratio_and_pad(
            item_mask,
            target_size=self.image_size,
            interpolation=TF.InterpolationMode.NEAREST,
            fill=0,
        )

        # 2. Optional small affine augmentation.
        # This keeps output size 512x512 and does not crop first.
        angle = random.uniform(-ROTATION_DEGREES, ROTATION_DEGREES)

        max_dx = TRANSLATE[0] * self.image_size
        max_dy = TRANSLATE[1] * self.image_size

        translations = (
            int(round(random.uniform(-max_dx, max_dx))),
            int(round(random.uniform(-max_dy, max_dy))),
        )

        scale = random.uniform(SCALE_RANGE[0], SCALE_RANGE[1])
        shear = [0.0, 0.0]

        img = TF.affine(
            img,
            angle=angle,
            translate=translations,
            scale=scale,
            shear=shear,
            interpolation=TF.InterpolationMode.BILINEAR,
            fill=255,
        )

        item_mask = TF.affine(
            item_mask,
            angle=angle,
            translate=translations,
            scale=scale,
            shear=shear,
            interpolation=TF.InterpolationMode.NEAREST,
            fill=0,
        )

        # 3. Drop mask BEFORE tensor conversion and normalization.
        # This makes blank-mask training match generated-image blank-mask inference.
        if random.random() < self.mask_dropout_prob:
            item_mask = Image.new("L", item_mask.size, 0)

        img_t = TF.to_tensor(img)
        mask_t = TF.to_tensor(item_mask)

        img_t = TF.normalize(img_t, GRAY_MEAN, GRAY_STD)
        mask_t = TF.normalize(mask_t, MASK_MEAN, MASK_STD)

        x = torch.cat([img_t, mask_t], dim=0)
        return x


class JointValTransform:
    """
    Validation transform:
    - Keeps full image visible
    - Keeps aspect ratio
    - Pads to 512x512
    - No crop
    - No random augmentation
    """

    def __init__(self, image_size):
        self.image_size = image_size

    def __call__(self, img, item_mask):
        img = resize_keep_ratio_and_pad(
            img,
            target_size=self.image_size,
            interpolation=TF.InterpolationMode.BILINEAR,
            fill=255,
        )

        item_mask = resize_keep_ratio_and_pad(
            item_mask,
            target_size=self.image_size,
            interpolation=TF.InterpolationMode.NEAREST,
            fill=0,
        )

        img_t = TF.to_tensor(img)
        mask_t = TF.to_tensor(item_mask)

        img_t = TF.normalize(img_t, GRAY_MEAN, GRAY_STD)
        mask_t = TF.normalize(mask_t, MASK_MEAN, MASK_STD)

        x = torch.cat([img_t, mask_t], dim=0)
        return x
# =========================
# DATASET
# =========================
class ProcessedSplitDataset_ItemMask(Dataset):
    def __init__(self, index_rows, processed_root, split, label_map, transform=None):
        self.processed_root = processed_root
        self.split = split.lower()
        self.transform = transform

        self.filepaths = []
        self.spatial_labels = []
        self.threat_labels = []
        self.has_item_polygon = []

        missing = []

        for r in index_rows:
            fp = (r.get("filepath") or "").replace("\\", "/").strip()
            row_split = (r.get("split") or "").strip().lower()

            if row_split != self.split:
                continue

            fname = Path(fp).name
            label = label_map.get(fp, label_map.get(fname, None))

            if label is None:
                missing.append(fp)
                continue

            img_path = self.processed_root / fp

            if not img_path.exists():
                missing.append(str(img_path))
                continue

            polygon_key = find_original_polygon_key(fname)

            self.filepaths.append(fp)
            self.spatial_labels.append(int(label["spatial"]))
            self.threat_labels.append(int(label["threat"]))
            self.has_item_polygon.append(polygon_key is not None)

        if missing:
            raise KeyError(f"Missing labels/files: {len(missing)} examples. First few: {missing[:10]}")

        logger.info(f"Dataset {split}: {len(self.filepaths)} samples loaded")
        logger.info(f"Dataset {split}: {sum(self.has_item_polygon)} samples have item polygons")
        logger.info(f"Dataset {split}: {len(self.has_item_polygon) - sum(self.has_item_polygon)} samples have blank item mask")

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img_path = self.processed_root / self.filepaths[idx]

        img = Image.open(img_path).convert("L")

        item_mask = create_item_only_mask(
            processed_filename=Path(self.filepaths[idx]).name,
            image_size=img.size,
        )

        if self.transform:
            x = self.transform(img, item_mask)
        else:
            img_t = TF.normalize(TF.to_tensor(img), GRAY_MEAN, GRAY_STD)
            mask_t = TF.normalize(TF.to_tensor(item_mask), MASK_MEAN, MASK_STD)
            x = torch.cat([img_t, mask_t], dim=0)

        spatial_y = torch.tensor(self.spatial_labels[idx], dtype=torch.long)
        threat_y = torch.tensor(self.threat_labels[idx], dtype=torch.long)

        return x, spatial_y, threat_y

# =========================
# CREATE DATASETS
# =========================
train_transform = JointTrainTransform(config.IMAGE_SIZE, mask_dropout_prob=get_progressive_mask_dropout(1))
val_transform = JointValTransform(config.IMAGE_SIZE)

train_ds = ProcessedSplitDataset_ItemMask(
    index_rows=index_rows,
    processed_root=config.PROCESSED_ROOT,
    split=config.TRAIN_SPLIT,
    label_map=train_label_map,
    transform=train_transform,
)

val_ds = ProcessedSplitDataset_ItemMask(
    index_rows=index_rows,
    processed_root=config.PROCESSED_ROOT,
    split="val",
    label_map=val_label_map,
    transform=val_transform,
)

logger.info(f"Train spatial distribution: {Counter(train_ds.spatial_labels)}")
logger.info(f"Train threat distribution:  {Counter(train_ds.threat_labels)}")
logger.info(f"Val spatial distribution:   {Counter(val_ds.spatial_labels)}")
logger.info(f"Val threat distribution:    {Counter(val_ds.threat_labels)}")

# =========================
# CLASS WEIGHTS
# =========================
def make_class_weights(labels, num_classes=2):
    counts = np.bincount(labels, minlength=num_classes)
    counts = np.maximum(counts, 1)

    weights = counts.sum() / (num_classes * counts)
    weights = np.sqrt(weights)
    weights = np.clip(weights, 0.85, 1.25)

    return torch.tensor(weights, dtype=torch.float32).to(device)

spatial_weights = make_class_weights(train_ds.spatial_labels, 2)
threat_weights = make_class_weights(train_ds.threat_labels, 2)

# Slightly increase contraband penalty
threat_weights[1] = threat_weights[1] * config.THREAT_CONTRABAND_WEIGHT_MULT

logger.info(f"spatial_weights: {spatial_weights.tolist()}")
logger.info(f"threat_weights: {threat_weights.tolist()}")

criterion_spatial = nn.CrossEntropyLoss(weight=spatial_weights, label_smoothing=0.03)
criterion_threat = nn.CrossEntropyLoss(weight=threat_weights, label_smoothing=0.02)

# =========================
# LOADERS
# =========================
train_loader = DataLoader(
    train_ds,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)

x, spatial_y, threat_y = next(iter(train_loader))
logger.info(f"Batch input shape: {x.shape}")
logger.info(f"Batch spatial label shape: {spatial_y.shape}")
logger.info(f"Batch threat label shape: {threat_y.shape}")

# =========================
# MODEL
# =========================
class SimpleCNN_MultiHead(nn.Module):
    def __init__(self, in_channels=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.shared_fc = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
        )

        self.spatial_head = nn.Linear(512, 2)
        self.threat_head = nn.Linear(512, 2)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.shared_fc(x)

        spatial_logits = self.spatial_head(x)
        threat_logits = self.threat_head(x)

        return spatial_logits, threat_logits

model = SimpleCNN_MultiHead(in_channels=2).to(device)

def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

logger.info(f"Total trainable parameters: {count_parameters(model):,}")

# =========================
# OPTIMIZER / SCHEDULER / SCALER
# =========================
optimizer = optim.AdamW(
    model.parameters(),
    lr=config.LR_FULL,
    weight_decay=config.WEIGHT_DECAY,
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    patience=config.SCHEDULER_PATIENCE,
    factor=config.SCHEDULER_FACTOR,
    min_lr=config.SCHEDULER_MIN_LR,
)

scaler = GradScaler("cuda") if config.USE_AMP and device.type == "cuda" else None

start_epoch = 1
best_score = -float("inf")
best_val_loss = float("inf")
epochs_without_improvement = 0

# =========================
# CHECKPOINT LOADING
# =========================
def safe_load_checkpoint(path: Path):
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)

def load_model_state_from_checkpoint(model, ckpt):
    if isinstance(ckpt, dict):
        if "model_state" in ckpt:
            model.load_state_dict(ckpt["model_state"], strict=True)
        elif "model_state_dict" in ckpt:
            model.load_state_dict(ckpt["model_state_dict"], strict=True)
        else:
            model.load_state_dict(ckpt, strict=True)
    else:
        raise ValueError("Checkpoint format is not supported.")

# =========================
# RESUME OR PRETRAIN
# =========================
if config.RESUME_TRAINING and config.RESUME_CHECKPOINT_PATH.exists():
    logger.info("=" * 80)
    logger.info(f"RESUME_TRAINING=True")
    logger.info(f"Continuing full training from: {config.RESUME_CHECKPOINT_PATH}")
    logger.info("=" * 80)

    ckpt = safe_load_checkpoint(config.RESUME_CHECKPOINT_PATH)

    try:
        load_model_state_from_checkpoint(model, ckpt)

        if isinstance(ckpt, dict):
            if ckpt.get("optimizer_state") is not None:
                optimizer.load_state_dict(ckpt["optimizer_state"])

            if ckpt.get("scheduler_state") is not None:
                scheduler.load_state_dict(ckpt["scheduler_state"])

            if scaler is not None and ckpt.get("scaler_state") is not None:
                scaler.load_state_dict(ckpt["scaler_state"])

            start_epoch = int(ckpt.get("epoch", 0)) + 1
            best_score = float(ckpt.get("best_score", -float("inf")))
            best_val_loss = float(ckpt.get("best_val_loss", float("inf")))

        logger.info(f"Resumed from epoch {start_epoch}")
        logger.info(f"Previous best score: {best_score:.4f}")
        logger.info(f"Previous best val loss: {best_val_loss:.4f}")

    except Exception as e:
        logger.warning(f"Could not resume because: {e}")
        logger.warning("Starting fresh training instead.")

elif config.PRETRAIN_FROM_PREVIOUS_BEST and config.PRETRAIN_CHECKPOINT_PATH.exists():
    logger.info("=" * 80)
    logger.info("PRETRAIN_FROM_PREVIOUS_BEST=True")
    logger.info(f"Loading model weights from: {config.PRETRAIN_CHECKPOINT_PATH}")
    logger.info("Optimizer/scheduler/scaler are reset.")
    logger.info("Training starts from epoch 1 with progressive mask dropout.")
    logger.info("=" * 80)

    ckpt = safe_load_checkpoint(config.PRETRAIN_CHECKPOINT_PATH)

    try:
        load_model_state_from_checkpoint(model, ckpt)

        start_epoch = 1

        if config.KEEP_PREVIOUS_BEST_SCORE and isinstance(ckpt, dict):
            best_score = float(ckpt.get("best_score", -float("inf")))
            best_val_loss = float(ckpt.get("best_val_loss", float("inf")))
        else:
            best_score = -float("inf")
            best_val_loss = float("inf")

        logger.info("Pretrained weights loaded successfully.")

    except Exception as e:
        logger.warning(f"Could not pretrain from previous best because: {e}")
        logger.warning("Starting fresh training instead.")
else:
    logger.info("=" * 80)
    logger.info("Starting training from scratch")
    logger.info("=" * 80)

# =========================
# LOSS HELPERS
# =========================
def compute_multihead_loss(spatial_logits, threat_logits, spatial_y, threat_y):
    loss_spatial = criterion_spatial(spatial_logits, spatial_y)
    loss_threat = criterion_threat(threat_logits, threat_y)
    return loss_spatial + loss_threat

def make_blank_mask_version(imgs):
    imgs_blank = imgs.clone()
    imgs_blank[:, 1:2, :, :] = BLANK_MASK_VALUE_NORM
    return imgs_blank

# =========================
# TRAIN / VALIDATE
# =========================
def train_one_epoch(model, loader, blank_consistency_weight):
    model.train()

    loss_sum = 0.0
    spatial_correct = 0
    threat_correct = 0
    both_correct = 0
    total = 0

    for imgs, spatial_y, threat_y in loader:
        imgs = imgs.to(device, non_blocking=True)
        spatial_y = spatial_y.to(device, non_blocking=True)
        threat_y = threat_y.to(device, non_blocking=True)

        imgs_blank = make_blank_mask_version(imgs)

        optimizer.zero_grad(set_to_none=True)

        if config.USE_AMP and scaler is not None:
            with autocast("cuda"):
                spatial_logits, threat_logits = model(imgs)
                loss_main = compute_multihead_loss(spatial_logits, threat_logits, spatial_y, threat_y)

                blank_spatial_logits, blank_threat_logits = model(imgs_blank)
                loss_blank = compute_multihead_loss(blank_spatial_logits, blank_threat_logits, spatial_y, threat_y)

                loss = loss_main + blank_consistency_weight * loss_blank

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_MAX_NORM)
            scaler.step(optimizer)
            scaler.update()

        else:
            spatial_logits, threat_logits = model(imgs)
            loss_main = compute_multihead_loss(spatial_logits, threat_logits, spatial_y, threat_y)

            blank_spatial_logits, blank_threat_logits = model(imgs_blank)
            loss_blank = compute_multihead_loss(blank_spatial_logits, blank_threat_logits, spatial_y, threat_y)

            loss = loss_main + blank_consistency_weight * loss_blank

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_MAX_NORM)
            optimizer.step()

        spatial_pred = spatial_logits.argmax(dim=1)
        threat_pred = threat_logits.argmax(dim=1)

        bs = imgs.size(0)
        loss_sum += loss.item() * bs
        spatial_correct += (spatial_pred == spatial_y).sum().item()
        threat_correct += (threat_pred == threat_y).sum().item()
        both_correct += ((spatial_pred == spatial_y) & (threat_pred == threat_y)).sum().item()
        total += bs

    return {
        "loss": loss_sum / max(total, 1),
        "spatial_acc": spatial_correct / max(total, 1),
        "threat_acc": threat_correct / max(total, 1),
        "both_acc": both_correct / max(total, 1),
    }


@torch.no_grad()
def validate(model, loader, force_blank_mask=False):
    model.eval()

    loss_sum = 0.0
    spatial_correct = 0
    threat_correct = 0
    both_correct = 0
    total = 0

    spatial_preds = []
    spatial_labels = []
    spatial_probs = []

    threat_preds = []
    threat_labels = []
    threat_probs = []

    for imgs, spatial_y, threat_y in loader:
        imgs = imgs.to(device, non_blocking=True)
        spatial_y = spatial_y.to(device, non_blocking=True)
        threat_y = threat_y.to(device, non_blocking=True)

        if force_blank_mask:
            imgs = make_blank_mask_version(imgs)

        spatial_logits, threat_logits = model(imgs)

        loss = compute_multihead_loss(spatial_logits, threat_logits, spatial_y, threat_y)

        spatial_prob = torch.softmax(spatial_logits, dim=1)
        threat_prob = torch.softmax(threat_logits, dim=1)

        spatial_pred = spatial_prob.argmax(dim=1)
        threat_pred = threat_prob.argmax(dim=1)

        bs = imgs.size(0)
        loss_sum += loss.item() * bs
        spatial_correct += (spatial_pred == spatial_y).sum().item()
        threat_correct += (threat_pred == threat_y).sum().item()
        both_correct += ((spatial_pred == spatial_y) & (threat_pred == threat_y)).sum().item()
        total += bs

        spatial_preds.extend(spatial_pred.cpu().numpy().tolist())
        spatial_labels.extend(spatial_y.cpu().numpy().tolist())
        spatial_probs.extend(spatial_prob.cpu().numpy().tolist())

        threat_preds.extend(threat_pred.cpu().numpy().tolist())
        threat_labels.extend(threat_y.cpu().numpy().tolist())
        threat_probs.extend(threat_prob.cpu().numpy().tolist())

    def binary_metrics(y, p, probs):
        precision, recall, f1, _ = precision_recall_fscore_support(
            y, p, average="binary", zero_division=0
        )

        try:
            auc = roc_auc_score(y, np.array(probs)[:, 1])
        except Exception:
            auc = 0.0

        cm = confusion_matrix(y, p, labels=[0, 1])

        return {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "auc": float(auc),
            "cm": cm.tolist(),
        }

    spatial_m = binary_metrics(spatial_labels, spatial_preds, spatial_probs)
    threat_m = binary_metrics(threat_labels, threat_preds, threat_probs)

    return {
        "loss": loss_sum / max(total, 1),
        "spatial_acc": spatial_correct / max(total, 1),
        "threat_acc": threat_correct / max(total, 1),
        "both_acc": both_correct / max(total, 1),
        "spatial_metrics": spatial_m,
        "threat_metrics": threat_m,
    }


def compute_single_score(metrics):
    both_acc = metrics["both_acc"]
    spatial_auc = metrics["spatial_metrics"]["auc"]
    threat_auc = metrics["threat_metrics"]["auc"]
    spatial_f1 = metrics["spatial_metrics"]["f1"]
    threat_f1 = metrics["threat_metrics"]["f1"]
    threat_recall = metrics["threat_metrics"]["recall"]

    score = (
        0.25 * both_acc +
        0.15 * spatial_auc +
        0.20 * threat_auc +
        0.10 * spatial_f1 +
        0.10 * threat_f1 +
        0.20 * threat_recall
    )

    return float(score)


def compute_selection_score(masked_metrics, blank_metrics):
    """
    This selects a model that works with masks AND without masks.

    blank_metrics has higher weight because generated images have no masks.
    """
    masked_score = compute_single_score(masked_metrics)
    blank_score = compute_single_score(blank_metrics)

    final_score = 0.40 * masked_score + 0.60 * blank_score
    return float(final_score), float(masked_score), float(blank_score)

# =========================
# TRAIN LOOP
# =========================
logger.info("=" * 80)
logger.info("Starting OPTIONAL-MASK SLOW TRAINING")
logger.info("Goal: model can use item mask if available, but does not depend on it")
logger.info("Input channels: 0=grayscale image, 1=item-only mask / blank mask")
logger.info("Train uses progressive mask dropout + blank-mask consistency loss")
logger.info("Validation saves best using masked validation + blank-mask validation")
logger.info("=" * 80)

for epoch in range(start_epoch, config.EPOCHS + 1):
    mask_dropout_prob = get_progressive_mask_dropout(epoch)
    blank_consistency_weight = get_blank_consistency_weight(epoch)

    train_transform.set_mask_dropout_prob(mask_dropout_prob)

    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        blank_consistency_weight=blank_consistency_weight,
    )

    val_masked_metrics = validate(model, val_loader, force_blank_mask=False)
    val_blank_metrics = validate(model, val_loader, force_blank_mask=True)

    selection_score, masked_score, blank_score = compute_selection_score(
        val_masked_metrics,
        val_blank_metrics,
    )

    val_loss = 0.40 * val_masked_metrics["loss"] + 0.60 * val_blank_metrics["loss"]

    scheduler.step(selection_score)

    current_lr = optimizer.param_groups[0]["lr"]

    if val_loss < best_val_loss:
        best_val_loss = val_loss

    checkpoint_dict = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict() if scaler is not None else None,

        "train_metrics": train_metrics,
        "val_masked_metrics": val_masked_metrics,
        "val_blank_metrics": val_blank_metrics,

        "selection_score": selection_score,
        "masked_score": masked_score,
        "blank_score": blank_score,
        "best_score": max(best_score, selection_score),
        "best_val_loss": best_val_loss,

        "spatial_classes": config.SPATIAL_CLASSES,
        "threat_classes": config.THREAT_CLASSES,
        "input_channels": ["grayscale_image", "optional_item_mask"],
        "item_mask_labels": ["shampoo", "blade"],
        "excluded_mask_labels": ["tray"],

        "seed": SEED,
        "image_size": config.IMAGE_SIZE,
        "resize_mode": "keep_aspect_ratio_pad_no_crop",
        "gray_mean": GRAY_MEAN,
        "gray_std": GRAY_STD,
        "mask_mean": MASK_MEAN,
        "mask_std": MASK_STD,
        "blank_mask_value_norm": BLANK_MASK_VALUE_NORM,

        "mask_dropout_prob": mask_dropout_prob,
        "blank_consistency_weight": blank_consistency_weight,
        "pretrained_from_previous_best": bool(
            config.PRETRAIN_FROM_PREVIOUS_BEST and config.PRETRAIN_CHECKPOINT_PATH.exists()
        ),
    }

    if selection_score > best_score:
        best_score = selection_score
        epochs_without_improvement = 0

        torch.save(model.state_dict(), config.BEST_MODEL_PATH)
        torch.save(checkpoint_dict, config.BEST_CHECKPOINT_PATH)

        logger.info(f"✓ New optional-mask best model saved: {config.BEST_MODEL_PATH}")
        logger.info(f"✓ New optional-mask best checkpoint saved: {config.BEST_CHECKPOINT_PATH}")
        logger.info(f"✓ New best selection score: {best_score:.4f}")
    else:
        epochs_without_improvement += 1

    logger.info("")
    logger.info("=" * 80)
    logger.info(f"Epoch [{epoch}/{config.EPOCHS}]")
    logger.info("=" * 80)

    logger.info(f"Mask dropout prob:        {mask_dropout_prob:.2f}")
    logger.info(f"Blank consistency weight: {blank_consistency_weight:.2f}")

    logger.info(f"Train Loss:        {train_metrics['loss']:.4f}")
    logger.info(f"Train Spatial Acc: {train_metrics['spatial_acc']:.3f}")
    logger.info(f"Train Threat Acc:  {train_metrics['threat_acc']:.3f}")
    logger.info(f"Train Both Acc:    {train_metrics['both_acc']:.3f}")

    logger.info("")
    logger.info("Validation with real item mask:")
    logger.info(f"  Loss:        {val_masked_metrics['loss']:.4f}")
    logger.info(f"  Spatial Acc: {val_masked_metrics['spatial_acc']:.3f}")
    logger.info(f"  Threat Acc:  {val_masked_metrics['threat_acc']:.3f}")
    logger.info(f"  Both Acc:    {val_masked_metrics['both_acc']:.3f}")
    logger.info(f"  Threat R:    {val_masked_metrics['threat_metrics']['recall']:.3f}")
    logger.info(f"  Threat F1:   {val_masked_metrics['threat_metrics']['f1']:.3f}")
    logger.info(f"  Threat AUC:  {val_masked_metrics['threat_metrics']['auc']:.3f}")
    logger.info(f"  Threat CM:   {val_masked_metrics['threat_metrics']['cm']}")

    logger.info("")
    logger.info("Validation with BLANK mask:")
    logger.info(f"  Loss:        {val_blank_metrics['loss']:.4f}")
    logger.info(f"  Spatial Acc: {val_blank_metrics['spatial_acc']:.3f}")
    logger.info(f"  Threat Acc:  {val_blank_metrics['threat_acc']:.3f}")
    logger.info(f"  Both Acc:    {val_blank_metrics['both_acc']:.3f}")
    logger.info(f"  Threat R:    {val_blank_metrics['threat_metrics']['recall']:.3f}")
    logger.info(f"  Threat F1:   {val_blank_metrics['threat_metrics']['f1']:.3f}")
    logger.info(f"  Threat AUC:  {val_blank_metrics['threat_metrics']['auc']:.3f}")
    logger.info(f"  Threat CM:   {val_blank_metrics['threat_metrics']['cm']}")

    logger.info("")
    logger.info(f"Masked score:      {masked_score:.4f}")
    logger.info(f"Blank score:       {blank_score:.4f}")
    logger.info(f"Selection Score:   {selection_score:.4f}")
    logger.info(f"Best Score:        {best_score:.4f}")
    logger.info(f"Best Val Loss:     {best_val_loss:.4f}")
    logger.info(f"Epochs no improve: {epochs_without_improvement}")
    logger.info(f"LR:                {current_lr:.8f}")

    if epochs_without_improvement >= config.EARLY_STOPPING_PATIENCE:
        logger.info("=" * 80)
        logger.info(f"Early stopping at epoch {epoch}")
        logger.info(f"Best selection score: {best_score:.4f}")
        logger.info(f"Best val loss: {best_val_loss:.4f}")
        logger.info("=" * 80)
        break

# =========================
# SAVE FINAL MODEL
# =========================
torch.save(model.state_dict(), config.MODEL_OUT_PATH)

final_checkpoint = {
    "epoch": epoch,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "scheduler_state": scheduler.state_dict(),
    "scaler_state": scaler.state_dict() if scaler is not None else None,

    "best_score": best_score,
    "best_val_loss": best_val_loss,

    "spatial_classes": config.SPATIAL_CLASSES,
    "threat_classes": config.THREAT_CLASSES,
    "input_channels": ["grayscale_image", "optional_item_mask"],
    "item_mask_labels": ["shampoo", "blade"],
    "excluded_mask_labels": ["tray"],

    "image_size": config.IMAGE_SIZE,
    "gray_mean": GRAY_MEAN,
    "gray_std": GRAY_STD,
    "mask_mean": MASK_MEAN,
    "mask_std": MASK_STD,
    "blank_mask_value_norm": BLANK_MASK_VALUE_NORM,

    "lr_full": config.LR_FULL,
    "weight_decay": config.WEIGHT_DECAY,
    "seed": SEED,
}

torch.save(final_checkpoint, config.FINAL_CHECKPOINT_PATH)

metrics_path = config.MODEL_DIR / "training_metrics_optional_mask.json"

with open(metrics_path, "w") as f:
    json.dump({
        "best_score": best_score,
        "best_val_loss": best_val_loss,

        "spatial_classes": config.SPATIAL_CLASSES,
        "threat_classes": config.THREAT_CLASSES,

        "input_channels": ["grayscale_image", "optional_item_mask"],
        "item_mask_labels": ["shampoo", "blade"],
        "excluded_mask_labels": ["tray"],

        "training_strategy": {
            "progressive_mask_dropout": True,
            "blank_mask_consistency_loss": True,
            "best_selection_uses_masked_and_blank_validation": True,
            "masked_validation_weight": 0.40,
            "blank_validation_weight": 0.60,
        },

        "image_size": config.IMAGE_SIZE,
        "gray_mean": GRAY_MEAN,
        "gray_std": GRAY_STD,
        "mask_mean": MASK_MEAN,
        "mask_std": MASK_STD,
        "blank_mask_value_norm": BLANK_MASK_VALUE_NORM,

        "lr_full": config.LR_FULL,
        "weight_decay": config.WEIGHT_DECAY,
        "seed": SEED,

        "best_model_path": str(config.BEST_MODEL_PATH),
        "best_checkpoint_path": str(config.BEST_CHECKPOINT_PATH),
        "final_model_path": str(config.MODEL_OUT_PATH),
        "final_checkpoint_path": str(config.FINAL_CHECKPOINT_PATH),
        "source_pretrain_checkpoint": str(config.SOURCE_PRETRAIN_CHECKPOINT_PATH),
    }, f, indent=2)

logger.info(f"Saved final model to: {config.MODEL_OUT_PATH}")
logger.info(f"Saved final checkpoint to: {config.FINAL_CHECKPOINT_PATH}")
logger.info(f"Saved metrics to: {metrics_path}")
logger.info(f"Best model path: {config.BEST_MODEL_PATH}")
logger.info(f"Best checkpoint path: {config.BEST_CHECKPOINT_PATH}")
logger.info("Training complete")